In [0]:
# Databricks notebook source
# =====================================================
# 1️⃣ Widgets
# =====================================================

import json
from pyspark.sql.functions import current_timestamp

start_time = spark.sql("SELECT current_timestamp()").collect()[0][0]

dbutils.widgets.text(
    "table_metadata",
    "{'table_id': '1', 'table_name': 'customers', 'source_system': 'sqlserver', "
    "'source_schema': 'banking', 'source_table': 'customers', "
    "'source_path': '', 'bronze_schema': 'bronze', "
    "'silver_schema': 'silver', 'active_flag': 'True', "
    "'load_order': '1', 'created_at': '2026-02-18 13:23:37.053711'}"
)

dbutils.widgets.text(
    "table_parameters",
    "{'load_type': 'MERGE', "
    "'primary_key': 'customer_id', "
    "'watermark_column': 'updated_at'}"
)

dbutils.widgets.text(
    "run_id",
    "472519629468310"
)
run_id=dbutils.widgets.get("run_id")

# Parse JSON safely
table_metadata = json.loads(dbutils.widgets.get("table_metadata").replace("'", '"'))
table_parameters = json.loads(dbutils.widgets.get("table_parameters").replace("'", '"'))

print("Table Metadata:", table_metadata)
print("Table Parameters:", table_parameters)
print(f"Run ID: {run_id}")

# COMMAND ----------

# =====================================================
# 2️⃣ Extract Variables
# =====================================================

table_id = int(table_metadata["table_id"])
table_name = table_metadata["table_name"]
source_system = table_metadata["source_system"].lower()
source_schema = table_metadata["source_schema"]
source_table = table_metadata["source_table"]
source_path = table_metadata["source_path"]
bronze_schema = table_metadata["bronze_schema"]

load_type = table_parameters.get("load_type")
watermark_column = table_parameters.get("watermark_column")

bronze_table_fqn = f"banking.{bronze_schema}.{table_name}"

print(f"Target Bronze Table: {bronze_table_fqn}")

# COMMAND ----------

# DBTITLE 1,Metadata Entry
# =====================================================
# Make and entry to audit table
# =====================================================
entry_exists = spark.sql(f"""
    SELECT 1
    FROM banking.metadata.pipeline_runs
    WHERE run_id = {run_id} AND table_id = {table_id}
""").count() > 0

if entry_exists:
    spark.sql(f"""
        UPDATE banking.metadata.pipeline_runs
        SET
            layer = 'Silver',
            start_time = TIMESTAMP('{start_time}'),
            end_time = NULL,
            status = 'INPROGRESS',
            number_of_records = NULL,
            error_message = NULL
        WHERE run_id = {run_id} AND table_id = {table_id}
    """)
else:
    spark.sql(f"""
        INSERT INTO banking.metadata.pipeline_runs
        VALUES (
            {run_id},
            {table_id},
            'Silver',
            TIMESTAMP('{start_time}'),
            NULL,  -- end time
            'INPROGRESS',
            NULL, --number of records
            NULL -- error message
        )
    """)

# COMMAND ----------

# =====================================================
# 3️⃣ Get Last Watermark (For Filtering Only)
# =====================================================

last_watermark = None

if load_type in ["APPEND", "MERGE"] and watermark_column:
    watermark_df = spark.sql(f"""
        SELECT last_watermark_value
        FROM banking.metadata.table_watermarks
        WHERE table_id = {table_id}
    """)
    
    if watermark_df.count() > 0:
        last_watermark = watermark_df.first()["last_watermark_value"]

print("Last Watermark:", last_watermark)

# COMMAND ----------

# MAGIC %sql
# MAGIC create schema if not exists banking.bronze

# COMMAND ----------

# =====================================================
# 4️⃣ Read Source
# =====================================================

try:
    if source_system == "sqlserver":

        # 🔐 Read connection JSON from secret
        secret_json = dbutils.secrets.get(
            scope="banking-scope",
            key="sqlserver-connection-json"
        )

        config = json.loads(secret_json)

        jdbc_url = f"jdbc:sqlserver://{config['host']}:{config['port']};database={config['database']}"

        jdbc_properties = {
            "user": config["user"],
            "password": config["password"],
            "driver": config["driver"]
        }

        # Build query
        if load_type in ["APPEND", "MERGE"] and last_watermark:
            query = f"""
            (SELECT * FROM {source_schema}.{source_table}
             WHERE {watermark_column} > '{last_watermark}') AS src
            """
        else:
            query = f"(SELECT * FROM {source_schema}.{source_table}) AS src"

        source_df = spark.read.jdbc(
            url=jdbc_url,
            table=query,
            properties=jdbc_properties
        )

    elif source_system == "blob":

        source_df = (
            spark.readStream
            .format("cloudFiles")
            .option("cloudFiles.format", "csv")
            .option(
                "cloudFiles.schemaLocation",
                f"/Volumes/banking/source/volume/_schema/{table_name}"
            )
            .option("header", "true")
            .load(source_path)
        )

    else:
        raise ValueError("Unsupported source_system")

    # =====================================================
    # 5️⃣ Add insert_timestamp
    # =====================================================

    source_df = source_df.withColumn("insert_timestamp", current_timestamp())

    # =====================================================
    # 6️⃣ Write to Bronze (Append Only)
    # =====================================================

    if source_system == "blob":

        (
            source_df.writeStream
            .format("delta")
            .option(
                "checkpointLocation",
                f"/Volumes/banking/source/volume/_checkpoints/{table_name}"
            )            
            .outputMode("append")
            .trigger(availableNow=True)
            .toTable(bronze_table_fqn)
        )

        records_read = None  # Streaming, can't count records here

    else:

        (
            source_df.write
            .format("delta")
            .mode("append")
            .saveAsTable(bronze_table_fqn)
        )

        records_read = source_df.count()

    print("Source → Bronze Load Completed Successfully.")
    print("Watermark will be updated after Silver load.")

except Exception as e:
    end_time = spark.sql("SELECT current_timestamp()").collect()[0][0]
    error_message = str(e)

    spark.sql(f"""
        UPDATE banking.metadata.pipeline_runs
        SET
            end_time = TIMESTAMP('{end_time}'),
            status = 'FAILED',
            error_message = {'NULL' if not error_message else "'" + error_message.replace("'", "") + "'"}
        WHERE table_id = {table_id} AND run_id = {run_id} 
    """)
    raise

In [0]:
# =====================================================
# 🧪 TEST: Blob Storage Source
# =====================================================

print("="*60)
print("🧪 TESTE: Blob Storage como fonte alternativa")
print("="*60)

# Configurar para usar Blob
test_source_system = "blob"
test_source_path = "/Volumes/banking/source/volume/customers/"
test_bronze_table = "banking.bronze.customers_blob_test"

try:
    print(f"\n📂 Verificando se o volume existe: {test_source_path}")
    
    # Tentar listar arquivos no volume
    try:
        files = dbutils.fs.ls(test_source_path)
        print(f"✅ Volume existe! {len(files)} arquivo(s) encontrado(s)")
        
        for file in files[:5]:  # Mostrar até 5 arquivos
            size_mb = file.size / (1024 * 1024)
            print(f"   📄 {file.name:<40} ({size_mb:.2f} MB)")
        
        if len(files) > 5:
            print(f"   ... e mais {len(files) - 5} arquivo(s)")
    
    except Exception as e:
        print(f"⚠️ Volume não existe ou está vazio: {str(e)[:100]}")
        print("\n💡 Para criar o volume e fazer upload:")
        print("   1. CREATE VOLUME IF NOT EXISTS banking.source.volume")
        print("   2. Upload arquivos CSV via UI ou código")
        print("\n⏭️ Pulando teste de leitura.")
        raise
    
    print(f"\n🔄 Tentando ler dados com Auto Loader...")
    
    # Tentar ler com Auto Loader (streaming)
    test_df = (
        spark.readStream
        .format("cloudFiles")
        .option("cloudFiles.format", "csv")
        .option(
            "cloudFiles.schemaLocation",
            f"/Volumes/banking/source/volume/_schema/customers_test"
        )
        .option("header", "true")
        .load(test_source_path)
    )
    
    print("✅ Auto Loader configurado com sucesso!")
    print(f"📋 Schema detectado: {len(test_df.columns)} colunas")
    print(f"   Colunas: {', '.join(test_df.columns[:10])}")
    
    if len(test_df.columns) > 10:
        print(f"   ... e mais {len(test_df.columns) - 10} colunas")
    
    # Adicionar timestamp
    test_df = test_df.withColumn("insert_timestamp", current_timestamp())
    
    print(f"\n💾 Gravando na tabela de teste: {test_bronze_table}")
    
    # Escrever na bronze (streaming)
    (
        test_df.writeStream
        .format("delta")
        .option(
            "checkpointLocation",
            f"/Volumes/banking/source/volume/_checkpoints/customers_test"
        )
        .outputMode("append")
        .trigger(availableNow=True)
        .toTable(test_bronze_table)
    )
    
    print("✅ Gravação concluída!")
    
    # Verificar quantos registros foram carregados
    record_count = spark.table(test_bronze_table).count()
    print(f"\n📊 Total de registros carregados: {record_count:,}")
    
    if record_count > 0:
        print("\n📄 Amostra dos dados (5 primeiras linhas):")
        display(spark.table(test_bronze_table).limit(5))
    
    print("\n" + "="*60)
    print("🎉 SUCESSO! Blob Storage está funcional!")
    print("="*60)
    print("\n💡 Agora você pode usar as duas fontes:")
    print("   • Azure SQL (primária): 4.000 registros")
    print("   • Blob Storage (backup): funcionando")
    
except Exception as e:
    print("\n" + "="*60)
    print("❌ TESTE FALHOU")
    print("="*60)
    print(f"Erro: {str(e)[:200]}")
    print("\n📋 Status: Apenas Azure SQL está disponível no momento.")
    print("   Para habilitar Blob Storage, é necessário:")
    print("   1. Criar o volume: banking.source.volume")
    print("   2. Fazer upload de arquivos CSV")

In [0]:
# =====================================================
# 📦 BACKUP: Exportar Azure SQL → CSV no Volume
# =====================================================

print("="*60)
print("📦 Criando backup CSV dos dados do Azure SQL")
print("="*60)

# Configurações
foreign_table = "azure_sql_banking_catalog.banking.customers"
output_path = "/Volumes/banking/source/volume/customers/"

try:
    print(f"\n🔄 Lendo dados do Azure SQL: {foreign_table}")
    
    # Ler dados do Azure SQL (batch, não streaming)
    df = spark.sql(f"SELECT * FROM {foreign_table}")
    
    row_count = df.count()
    print(f"✅ Lidos {row_count:,} registros do Azure SQL")
    print(f"📋 Schema: {len(df.columns)} colunas")
    
    # Mostrar primeiras colunas
    print("\n📊 Colunas (primeiras 10):")
    for col in df.columns[:10]:
        print(f"   • {col}")
    if len(df.columns) > 10:
        print(f"   ... e mais {len(df.columns) - 10} colunas")
    
    print(f"\n💾 Salvando CSV no volume: {output_path}")
    
    # Salvar como CSV (1 arquivo apenas para facilitar)
    (
        df.coalesce(1)
        .write
        .mode("overwrite")
        .option("header", "true")
        .csv(output_path)
    )
    
    print("✅ CSV salvo com sucesso!")
    
    # Listar arquivos criados
    files = dbutils.fs.ls(output_path)
    print(f"\n📂 Arquivos criados ({len(files)} total):")
    for file in files:
        size_mb = file.size / (1024 * 1024)
        icon = "📄" if file.name.endswith(".csv") else "📁"
        print(f"   {icon} {file.name:<50} ({size_mb:.2f} MB)")
    
    print("\n" + "="*60)
    print("🎉 SUCESSO! Backup CSV criado!")
    print("="*60)
    print("\n💡 Agora você tem:")
    print("   ✅ Fonte primária: Azure SQL (4.000 registros)")
    print("   ✅ Fonte backup: CSV no volume (4.000 registros)")
    print("\n🔄 Próximo passo: implementar fallback automático!")
    
except Exception as e:
    print("\n" + "="*60)
    print("❌ ERRO AO CRIAR BACKUP")
    print("="*60)
    print(f"Erro: {str(e)[:300]}")
    import traceback
    print("\n" + traceback.format_exc()[:500])

# 🔄 Sistema de Fallback Automático

## ✨ Como Funciona

O pipeline agora tem **duas fontes de dados** com fallback automático:

1. **Fonte Primária:** Azure SQL (via Lakehouse Federation)
2. **Fonte Backup:** CSV no Blob Storage (`/Volumes/banking/source/volume/customers/`)

### 👉 Comportamento:

- 🔄 **Tenta Azure SQL primeiro**
- ⚠️ **Se falhar** (firewall, conexão, credenciais):
  - 🛡️ Ativa fallback automaticamente
  - 📄 Lê dados do CSV backup
  - ✅ Pipeline continua sem interrupção

---

## 🛠️ Configurando a Fonte

### Modo 1: Azure SQL como Primária (Recomendado)

```python
table_metadata = {
    'source_system': 'sqlserver',
    'source_schema': 'banking',
    'source_table': 'customers',
    'source_path': '',  # Não usado
    ...
}
```

**Resultado:**
- ✅ Usa Azure SQL
- 🛡️ Se falhar → CSV backup

---

### Modo 2: CSV Blob como Primária

```python
table_metadata = {
    'source_system': 'blob',
    'source_schema': '',  # Não usado
    'source_table': '',   # Não usado
    'source_path': '/Volumes/banking/source/volume/customers/',
    ...
}
```

**Resultado:**
- 📄 Usa CSV diretamente
- ⚠️ Sem fallback (CSV é a única fonte)

---

## 📊 Monitoramento

O pipeline registra qual fonte foi usada:

```
✅ Fonte usada: Azure SQL
✅ Fonte usada: CSV Blob (Fallback)  ← Fallback ativado!
```

---

## 🔧 Manutenção do Backup CSV

### Atualizar Backup CSV:

Execute a célula **"📦 Export Azure SQL → CSV Backup"** para:
- Exportar dados atuais do Azure SQL
- Substituir CSV antigo no volume
- Manter backup sincronizado

**Frequência sugerida:** Diário ou semanal

---

## ✅ Vantagens do Sistema

✅ **Alta Disponibilidade:** Pipeline continua mesmo se Azure SQL cair  
✅ **Transparência:** Registra qual fonte foi usada  
✅ **Automático:** Nenhuma intervenção manual necessária  
✅ **Resiliente:** Protege contra falhas de firewall, rede, credenciais  
✅ **Flexível:** Pode usar CSV como fonte primária se necessário  

---

## 🚨 Quando o Fallback Ativa?

- 🚫 Firewall do Azure SQL bloqueia IP do Databricks
- 🔒 Credenciais inválidas ou expiradas
- 🌐 Problemas de rede entre Databricks e Azure
- ⚡ Azure SQL indisponível (manutenção, outage)
- 📉 Catalog/connection do Lakehouse Federation com problema

---

## 📝 Exemplo de Execução Normal

```
🔄 Tentando fonte primária: Azure SQL...
✅ Sucesso! 4,000 registros do Azure SQL

============================================================
✅ Source → Bronze Load Completed Successfully!
============================================================
📊 Fonte usada: Azure SQL
📄 Registros carregados: 4,000
============================================================
```

---

## 📝 Exemplo de Execução com Fallback

```
🔄 Tentando fonte primária: Azure SQL...
⚠️ Falha no Azure SQL: Client with IP address 'X.X.X.X' is not allowed...
🔄 Ativando fallback: CSV Blob Storage...
✅ Fallback ativado! Lendo de: /Volumes/banking/source/volume/customers/

============================================================
✅ Source → Bronze Load Completed Successfully!
============================================================
📊 Fonte usada: CSV Blob (Fallback)
⚠️  Fallback ativado! Erro primário: Client with IP address...
🛡️ Sistema resiliente: dados carregados do backup CSV
============================================================
```

In [0]:
# =====================================================
# 🔄 RESET: Dropar tabela Bronze para testar CDC
# =====================================================

print("="*60)
print("🔄 Resetando tabela Bronze para teste de CDC")
print("="*60)

bronze_table = "banking.bronze.customers"

try:
    # Verificar se tabela existe
    if spark.catalog.tableExists(bronze_table):
        print(f"\n🗑️ Dropando tabela existente: {bronze_table}")
        spark.sql(f"DROP TABLE IF EXISTS {bronze_table}")
        print("✅ Tabela dropada com sucesso!")
    else:
        print(f"\nℹ️ Tabela {bronze_table} não existe ainda")
    
    print("\n" + "="*60)
    print("✅ Reset concluído! Pronto para teste de CDC")
    print("="*60)
    print("\n💡 Próximo passo:")
    print("   1. Execute a célula principal (Cell 1)")
    print("   2. Primeira execução: criará tabela com 4.000 registros")
    print("   3. Segunda execução: fará MERGE (CDC) de novos/modificados")
    print("   4. Registros serão atualizados/inseridos, não sobrescritos!")
    
except Exception as e:
    print("\n" + "="*60)
    print("❌ ERRO AO RESETAR")
    print("="*60)
    print(f"Erro: {str(e)[:300]}")

In [0]:
# =====================================================
# 🔄 VERSÃO SIMPLES: Azure SQL com Fallback para Blob
# =====================================================
# Esta é uma versão simplificada do código original
# com fallback automático para Blob Storage
# =====================================================

import json
from pyspark.sql.functions import current_timestamp

# Configurações (simular widgets)
table_name = "customers"
source_schema = "banking"
source_table = "customers"
bronze_table_fqn = "banking.bronze.customers"
watermark_column = "updated_at"
last_watermark = None  # Para teste, sem filtro incremental

print("="*60)
print("🔄 Pipeline com Fallback Automático")
print("="*60)

source_used = None
fallback_activated = False

try:
    # =====================================================
    # PASSO 1: Tentar Azure SQL
    # =====================================================
    try:
        print("\n🔄 Tentando Azure SQL...")
        
        foreign_table = f"azure_sql_banking_catalog.{source_schema}.{source_table}"
        
        # Build WHERE clause
        where_clause = ""
        if last_watermark:
            where_clause = f"WHERE {watermark_column} > TIMESTAMP('{last_watermark}')"
        
        # CREATE OR REPLACE (versão original)
        spark.sql(f"""
            CREATE OR REPLACE TABLE {bronze_table_fqn}
            USING DELTA
            AS
            SELECT *, CURRENT_TIMESTAMP() as insert_timestamp
            FROM {foreign_table}
            {where_clause}
        """)
        
        records_read = spark.table(bronze_table_fqn).count()
        source_used = "Azure SQL"
        print(f"✅ Sucesso! {records_read:,} registros do Azure SQL")
        
    except Exception as azure_error:
        # =====================================================
        # PASSO 2: FALLBACK para Blob Storage
        # =====================================================
        print(f"\n⚠️ Azure SQL falhou: {str(azure_error)[:100]}")
        print("🔄 Ativando fallback: Blob Storage...")
        fallback_activated = True
        
        # Ler do Blob Storage
        blob_path = f"/Volumes/banking/source/volume/{table_name}/"
        
        source_df = (
            spark.readStream
            .format("cloudFiles")
            .option("cloudFiles.format", "csv")
            .option(
                "cloudFiles.schemaLocation",
                f"/Volumes/banking/source/volume/_schema/{table_name}"
            )
            .option("header", "true")
            .load(blob_path)
        )
        
        # Adicionar timestamp
        source_df = source_df.withColumn("insert_timestamp", current_timestamp())
        
        # Escrever na Bronze
        (
            source_df.writeStream
            .format("delta")
            .option(
                "checkpointLocation",
                f"/Volumes/banking/source/volume/_checkpoints/{table_name}"
            )
            .outputMode("append")
            .trigger(availableNow=True)
            .toTable(bronze_table_fqn)
        )
        
        source_used = "Blob Storage (Fallback)"
        print(f"✅ Fallback ativado! Dados carregados do: {blob_path}")
    
    # =====================================================
    # Resultado Final
    # =====================================================
    print("\n" + "="*60)
    print("✅ SUCESSO!")
    print("="*60)
    print(f"📊 Fonte usada: {source_used}")
    
    if fallback_activated:
        print("⚠️ Fallback foi ativado (Azure SQL não disponível)")
        print("🛡️ Sistema resiliente: dados carregados do backup CSV")
    
    # Verificar total de registros
    total = spark.table(bronze_table_fqn).count()
    print(f"📄 Total na Bronze: {total:,} registros")
    print("="*60)
    
except Exception as e:
    print("\n" + "="*60)
    print("❌ ERRO FATAL")
    print("="*60)
    print(f"Erro: {str(e)[:300]}")
    raise